# Feature Generation Reproduction

This notebook reproduces the columns in `features_df` using the `generate_features` function from `core/features.py`.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import sys
import os

# Ensure the project root is in the path so we can import core
if '..' not in sys.path:
    sys.path.append(os.path.abspath('.'))

from core.features import generate_features
from core.settings import GLOBAL_SETTINGS

## 1. Create Sample Data

We will generate a synthetic MultiIndex DataFrame for 'GOOG' matching the structure provided.

In [ ]:
def create_sample_ohlcv(ticker="GOOG", start="2004-08-19", end="2026-04-22"):
    dates = pd.date_range(start=start, end=end, freq='B')
    n = len(dates)
    
    # Generate synthetic price path
    np.random.seed(42)
    returns = np.random.normal(0.0005, 0.01, n)
    price = 100 * np.exp(np.cumsum(returns))
    
    df = pd.DataFrame({
        "Adj Open": price * (1 - 0.001),
        "Adj High": price * (1 + 0.005),
        "Adj Low": price * (1 - 0.005),
        "Adj Close": price,
        "Volume": np.random.randint(1000000, 10000000, n)
    }, index=dates)
    
    # Create MultiIndex
    df.index.name = "Date"
    df["Ticker"] = ticker
    df = df.reset_index().set_index(["Ticker", "Date"])
    
    return df

df_ohlcv = create_sample_ohlcv()
print("Sample Data Info:")
df_ohlcv.info()

## 2. Generate Features

Now we run the `generate_features` function. Note that we provide `df_ohlcv` and optionally `benchmark_ticker`. If `df_indices` and `df_fed` are omitted, they default to `None` and the function handles it.

In [ ]:
features_df, macro_df = generate_features(
    df_ohlcv=df_ohlcv,
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"], # Usually 'SPY'
)

print("\nFeatures DataFrame Columns:")
print(features_df.columns.tolist())

print("\nFirst 5 rows of features_df:")
display(features_df.head())

## 3. Verification

Verify that the columns match the expected feature set.

In [ ]:
expected_cols = [
    "ATR", "ATRP", "TRP", "RSI", "Mom_21", "Consistency", "IR_63", 
    "Beta_63", "DD_21", "AutoCorr_15", "Ret_1d", "Range_Pos_20", 
    "Slope_P_5", "Slope_V_5", "Convexity", "RollingStalePct", 
    "RollMedDollarVol", "RollingSameVolCount"
]

missing = [c for c in expected_cols if c not in features_df.columns]
if not missing:
    print("✅ All expected columns are present!")
else:
    print(f"❌ Missing columns: {missing}")